**`enrich_parcels_with_places_fmv_conus`**

Link parcel data from `openplaces` to `places-fmv-conus`.

## Load ingested parcels from `openplaces`
Example: Buncombe county, NC

In [ ]:
from openplaces.api import read_entities

ADMIN_ID = 'US-NC-BO'

PARCEL_RECIPE_ID = 'US-NC_parcel-nconemap-2025'

parcels = read_entities(PARCEL_RECIPE_ID, ADMIN_ID, geom=True)
print('Columns:', ', '.join(parcels.columns))
parcels.sample(5).iloc[:, :5]

In [ ]:
from openplaces.api import get_admin

admin3 = get_admin(ADMIN_ID, geom=True)

## Load parcel predictors from `places-fmv-conus`
Download: https://drive.google.com/drive/folders/1f2fqT4bdnHbwh80vPiGdnCJ8UV-X_PQb

In [ ]:
import pandas as pd
from openplaces.path import external_path

PARQUET_PATH = external_path(
    'US', 'parcel-placesfmv-2026', filename='37021_parcel_predictors.parquet'
)
print('File:', PARQUET_PATH)
parcel_predictors = pd.read_parquet(PARQUET_PATH)
print('Columns:', ', '.join(parcel_predictors.columns))
parcel_predictors.sample(5)

## Load parcel boundaries from `places-fmv-conus`
To test alternative parcel IDs

In [ ]:
import geopandas as gpd
from openplaces.path import external_path

PARQUET_PATH = external_path(
    'US', 'parcel-placesfmv-2026', filename='37021_parcel_boundaries.parquet'
)
parcel_boundaries = gpd.read_parquet(PARQUET_PATH)

## Compute linkage IDs

In [ ]:
from openplaces.geo.ids import get_geo_ids

parcel_predictors['geo_id'] = get_geo_ids(parcel_boundaries)

# Test different linkages
Conclusion:

* Use 'geo_id'.
* For parcels that don't match by 'geo_id': write crosswalk
* Where should the crosswalk go? In which direction? Always moving forward? Always backwards-compatible?

In [ ]:
GEO_IDS = ['geo_id']

In [ ]:
for geo_id in GEO_IDS:
    print(geo_id)
    if parcel_predictors.reset_index()[geo_id].duplicated().any():
        raise ValueError(f'Duplicate {geo_id}.')
    joined_parcels = parcels[~parcels[geo_id].duplicated()].join(
        parcel_predictors.reset_index().set_index(geo_id),
        on=geo_id,
        lsuffix='_openplaces',
        rsuffix='_places',
    )
    mask_notnull = joined_parcels['x'].notnull()
    print(f'{mask_notnull.mean():.1%} joined')
    if not mask_notnull.any():
        print('Skipped')
        continue

    ax = joined_parcels.plot(
        # 'fld_fr_fath_f100',
        # cmap='Blues',
        'slope',
        scheme='quantiles',
        cmap='RdYlBu_r',
        legend=True,
        legend_kwds={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
    )
    admin3.boundary.plot(ax=ax, color='black', linewidth=0.1)
    ax.axis('off')
    ax.set_title(f'Parcels linked via {geo_id}')